# Kronos PoC: Kronos-small vs GBM on Japanese Stocks

Spec: `docs/superpowers/specs/2026-05-02-kronos-poc-design.md`. Run all cells top to bottom. See `notebooks/README.md` for details.

## 1. Setup

In [ ]:
# Cell 1: setup
!pip install -q einops==0.8.1 huggingface_hub==0.33.1 safetensors==0.6.2 tqdm==4.67.1
# torch / numpy / pandas / matplotlib are pre-installed on Colab; do not pin to avoid breaking Colab base env.

import os, json, time, hashlib, pickle, getpass
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from tqdm.auto import tqdm

print(f"torch={torch.__version__}, cuda={torch.cuda.is_available()}, mps={torch.backends.mps.is_available()}")

## 2. Auth

In [ ]:
# Cell 2: auth
# Try Colab Secrets first, fall back to interactive getpass. The key never gets printed.
try:
    from google.colab import userdata
    JQUANTS_API_KEY = userdata.get("JQUANTS_API_KEY")
    print("Loaded JQUANTS_API_KEY from Colab Secrets.")
except Exception:
    JQUANTS_API_KEY = None

if not JQUANTS_API_KEY:
    JQUANTS_API_KEY = getpass.getpass("JQuants API key (Premium plan): ").strip()

assert JQUANTS_API_KEY, "API key required"
os.environ["JQUANTS_API_KEY"] = JQUANTS_API_KEY
print(f"API key loaded ({len(JQUANTS_API_KEY)} chars). Header will be sent as 'x-api-key: ****'.")

## 3. Fetch OHLCV (~4 years)

In [ ]:
# Cell 3: fetch_data
STOCKS = {"7203": "Toyota", "6758": "Sony Group", "9984": "SoftBank Group"}
JQUANTS_BASE = "https://api.jquants.com/v2"
LOOKBACK = 256
HORIZON = 20
N_WINDOWS = 36
STRIDE = 21
SAMPLE_COUNT = 30
# Calendar-day budget for ~1011 business days (allow 50% headroom for weekends/holidays).
FETCH_DAYS_CAL = int((LOOKBACK + (N_WINDOWS - 1) * STRIDE + HORIZON) * 1.5) + 30

def fetch_ohlcv(code4: str) -> pd.DataFrame:
    """Fetch ~4 years of daily OHLCV from JQuants v2. Returns DataFrame indexed by Date with O/H/L/C/V columns."""
    code5 = code4 + "0"  # 4-digit -> 5-digit
    today = datetime.utcnow().date()
    start = (today - pd.Timedelta(days=FETCH_DAYS_CAL)).isoformat()
    end = today.isoformat()
    url = f"{JQUANTS_BASE}/equities/bars/daily"
    params = {"code": code5, "from": start, "to": end}
    headers = {"x-api-key": os.environ["JQUANTS_API_KEY"]}
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            data = r.json().get("data", [])
            if not data:
                raise RuntimeError(f"No data returned for {code4}")
            df = pd.DataFrame(data)
            df["Date"] = pd.to_datetime(df["Date"])
            df = df.set_index("Date").sort_index()
            df = df.rename(columns={"AdjO": "open", "AdjH": "high", "AdjL": "low", "AdjC": "close", "AdjVo": "volume"})
            return df[["open", "high", "low", "close", "volume"]].astype(float)
        except requests.HTTPError as e:
            if e.response.status_code in (401, 403):
                raise RuntimeError("JQuants auth failed — check API key and Premium plan status") from e
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("unreachable")

df_dict = {}
for code in STOCKS:
    df_dict[code] = fetch_ohlcv(code)
    print(f"{code} {STOCKS[code]}: rows={len(df_dict[code])}, range={df_dict[code].index.min().date()}..{df_dict[code].index.max().date()}")
    time.sleep(0.5)

# Sanity: enough rows for the full window plan
need = LOOKBACK + (N_WINDOWS - 1) * STRIDE + HORIZON
for code, df in df_dict.items():
    assert len(df) >= need, f"{code}: have {len(df)} rows, need >= {need}"
print(f"All stocks have enough rows (need >= {need}).")

## 4. Load Kronos

In [ ]:
# Cell 4: load_kronos
import sys, subprocess, pathlib

# Kronos repo is not on PyPI. Shallow-clone for the model wrapper code.
kronos_dir = pathlib.Path("/content/Kronos")
if not kronos_dir.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/shiyu-coder/Kronos.git", str(kronos_dir)])
if str(kronos_dir) not in sys.path:
    sys.path.insert(0, str(kronos_dir))

from model import Kronos, KronosTokenizer, KronosPredictor  # noqa: E402

# Device priority: CUDA (Colab T4) > MPS (unlikely on Colab) > CPU
if torch.cuda.is_available():
    device = "cuda:0"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-small")
predictor = KronosPredictor(model, tokenizer, device=device, max_context=512)
print(f"Kronos-small loaded on {device}.")

## 4.5. Smoke test

In [ ]:
# Cell 4.5: smoke_test
test_code = "7203"
test_df = df_dict[test_code].iloc[-(LOOKBACK + HORIZON + 4):-(HORIZON + 4)]  # any window with both lookback & horizon
test_horizon_ts = pd.Series(df_dict[test_code].iloc[-(HORIZON + 4):-4].index)
test_x_ts = pd.Series(test_df.index)
assert len(test_df) == LOOKBACK and len(test_horizon_ts) == HORIZON

x_df = test_df.copy()
x_df["amount"] = 0.0  # Kronos optional column
pred = predictor.predict(
    df=x_df[["open", "high", "low", "close", "volume", "amount"]],
    x_timestamp=test_x_ts,
    y_timestamp=test_horizon_ts,
    pred_len=HORIZON,
    T=1.0,
    top_p=0.9,
    sample_count=2,
    verbose=False,
)
assert pred.shape[0] == HORIZON, f"expected {HORIZON} rows, got {pred.shape}"
assert not pred["close"].isna().any(), "NaN in close"
assert (pred["close"] > 0).all(), "non-positive close"
print(f"Smoke OK: pred shape={pred.shape}, close range=[{pred['close'].min():.0f}, {pred['close'].max():.0f}]")

## 5. Walk-forward (Kronos)

In [ ]:
# Cell 5a: helpers (window generation + atomic pickle)
def window_origins(df: pd.DataFrame) -> list:
    """Return 36 window origins per spec §4.1.

    For each i in 0..35: origin_i index = T_end_idx - HORIZON - (35-i)*STRIDE
    Returns list of (origin_idx, lookback_dates, horizon_dates, S0, actual).
    """
    T_end_idx = len(df) - 1
    out = []
    for i in range(N_WINDOWS):
        origin_idx = T_end_idx - HORIZON - (N_WINDOWS - 1 - i) * STRIDE
        lo_lo = origin_idx - (LOOKBACK - 1)  # inclusive 256-bar window ending at origin_idx
        if lo_lo < 0 or origin_idx + HORIZON > T_end_idx:
            raise RuntimeError(f"window {i} out of range: lo_lo={lo_lo}, origin={origin_idx}, T_end={T_end_idx}")
        lookback_slice = df.iloc[lo_lo: origin_idx + 1]      # 256 rows including origin_idx
        horizon_slice = df.iloc[origin_idx + 1: origin_idx + 1 + HORIZON]
        assert len(lookback_slice) == LOOKBACK
        assert len(horizon_slice) == HORIZON
        out.append({
            "origin": lookback_slice.index[-1],
            "lookback": lookback_slice,
            "horizon_dates": list(horizon_slice.index),
            "actual": horizon_slice["close"].to_numpy(dtype=np.float64),
            "S0": float(lookback_slice["close"].iloc[-1]),
        })
    return out

def atomic_pickle_dump(obj, path: str):
    """Write pickle atomically: write to .tmp, fsync, rename. See spec §5.2."""
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

# self-check on first stock
_check = window_origins(df_dict["7203"])
assert len(_check) == N_WINDOWS
assert _check[-1]["horizon_dates"][-1] == df_dict["7203"].index[-1], "last window must end at T_end"
print(f"window_origins OK: 36 windows, last horizon ends at {_check[-1]['horizon_dates'][-1].date()}")

In [ ]:
# Cell 5: walkforward_kronos
KRONOS_RESULTS_PATH = "/content/kronos_results.pkl"

if os.path.exists(KRONOS_RESULTS_PATH):
    with open(KRONOS_RESULTS_PATH, "rb") as f:
        kronos_results = pickle.load(f)
    print(f"Resumed: {len(kronos_results)} windows already done")
else:
    kronos_results = []
done_keys = {(r["code"], r["lookback_end"]) for r in kronos_results}

todo = []
for code, df in df_dict.items():
    for w in window_origins(df):
        if (code, w["origin"]) in done_keys:
            continue
        todo.append((code, w))

for code, w in tqdm(todo, desc="Kronos walk-forward"):
    x_df = w["lookback"].copy()
    x_df["amount"] = 0.0
    x_ts = pd.Series(w["lookback"].index)
    y_ts = pd.Series(w["horizon_dates"])
    # IMPORTANT: KronosPredictor.predict() with sample_count>1 averages internally
    # (kronos.py:467 `preds = np.mean(preds, axis=1)`), returning a single averaged DataFrame.
    # For percentile metrics we need raw samples, so call predict() sample_count times
    # with sample_count=1 each. This is ~30x more Python overhead per window than a
    # batched no-mean variant would be, but it's the simplest way without forking Kronos.
    paths = np.zeros((SAMPLE_COUNT, HORIZON), dtype=np.float64)
    for s in range(SAMPLE_COUNT):
        single = predictor.predict(
            df=x_df[["open", "high", "low", "close", "volume", "amount"]],
            x_timestamp=x_ts, y_timestamp=y_ts, pred_len=HORIZON,
            T=1.0, top_p=0.9, sample_count=1, verbose=False,
        )
        paths[s, :] = single["close"].to_numpy(dtype=np.float64)
    kronos_results.append({
        "code": code,
        "lookback_end": w["origin"],
        "horizon_dates": w["horizon_dates"],
        "actual": w["actual"],
        "predicted_paths": paths,
        "S0": w["S0"],
    })
    atomic_pickle_dump(kronos_results, KRONOS_RESULTS_PATH)

print(f"Kronos walk-forward complete: {len(kronos_results)} windows total")